In [1]:
%matplotlib module://ipympl.backend_nbagg
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.ndimage import maximum_filter

import sys
sys.path.insert(0, "/workspace/segmentation")
from tools import img_show3D, window, img_show2D, img_show3D_interactive, img_compare3D


In [2]:
img_dir = Path ("/workspace/segmentation/data/practice/image_quadra/QUADRA_HC_021/retest_CT-AC.nii.gz")
img = sitk.ReadImage(img_dir)
img_vec = sitk.GetArrayFromImage(img)

mask_dir = Path ("/workspace/segmentation/data/practice/mask_quadra/QUADRA_HC_021/retest/liver.nii.gz")
mask = sitk.ReadImage(mask_dir)
mask_vec = sitk.GetArrayFromImage(mask)

In [5]:
spacingxyz = img.GetSpacing()

In [7]:
def harris_response_3d(image, spacing_xyz_mm, scale_mm, harris_k):
    """Return a physical-scale 3D Harris response.

    The structure tensor uses Gaussian derivatives at ``scale_mm`` and an
    integration scale 1.5 times larger. The 3D response is
    ``det(M) - k * trace(M)^3``.
    """

    from scipy.ndimage import gaussian_filter

    spacing = np.asarray(spacing_xyz_mm, dtype=np.float64)
    derivative_sigma = np.maximum(float(scale_mm) / spacing, 0.5)
    integration_sigma = np.maximum(1.5 * float(scale_mm) / spacing, 0.5)
    smoothed = gaussian_filter(
        np.asarray(image, dtype=np.float32), derivative_sigma, mode="nearest"
    )
    gradients = np.gradient(smoothed, *spacing, edge_order=1)
    gx, gy, gz = [np.asarray(component, dtype=np.float32) for component in gradients]
    a = gaussian_filter(gx * gx, integration_sigma, mode="nearest")
    d = gaussian_filter(gy * gy, integration_sigma, mode="nearest")
    f = gaussian_filter(gz * gz, integration_sigma, mode="nearest")
    b = gaussian_filter(gx * gy, integration_sigma, mode="nearest")
    c = gaussian_filter(gx * gz, integration_sigma, mode="nearest")
    e = gaussian_filter(gy * gz, integration_sigma, mode="nearest")
    determinant = a * d * f + 2.0 * b * c * e - a * e * e - d * c * c - f * b * b
    trace = a + d + f
    response = determinant - float(harris_k) * trace * trace * trace
    #np.asarray(np.maximum(response, 0.0), dtype=np.float32)
    return response, a,b,c,d,e,f,determinant,trace

In [11]:
response, a,b,c,d,e,f,determinant,trace = harris_response_3d(img_vec, spacing_xyz_mm= spacingxyz, scale_mm= 1, harris_k= 0.005)

In [14]:
def negative_check(array):
    has_negative = (array < 0).any()  # Or: np.any(array < 0)
    print(has_negative)

In [16]:
negative_check(response)
negative_check(a)
negative_check(d)
negative_check(f)

negative_check(b)
negative_check(c)
negative_check(e)

negative_check(determinant)
negative_check(trace)

True
False
False
False
True
True
True
True
False
